# Implement KL Divergence Loss — Solution

**Difficulty**: 🟡 Medium

---

### Problem Statement

KL divergence is everywhere in modern ML: the VAE regularizer, the knowledge-distillation loss, the KL penalty in RLHF/PPO, the `β·KL` term in GRPO, the trust region in TRPO. Interviewers love it because a "one-line" function hides a lot of decisions:

- **Which direction?** KL(P‖Q) ≠ KL(Q‖P), and the choice changes what a model learns.
- **What do you have?** Probabilities, logits, log-probs, Gaussian parameters, or only *samples*.
- **Is it numerically stable?** `0 · log 0`, softmax underflow, `-inf` logits.
- **How do you reduce?** Over classes, the batch, tokens, and padding.
- **For RL: is the gradient right?** The gradient of an estimator *used as a loss* is not always the gradient of KL.

$$\mathrm{KL}(P\,\|\,Q) = \sum_x P(x)\log\frac{P(x)}{Q(x)} = \mathbb{E}_{x\sim P}\big[\log P(x) - \log Q(x)\big] \;\ge\; 0$$

It is zero iff P = Q, it is **not symmetric**, and it is not a metric.

This notebook covers every variant you're likely to be asked about, in nine short parts. Each part is one function followed by a test cell.

| Part | Function | What it practises |
|---|---|---|
| 1 | `kl_categorical(p, q)` | The definition; `0·log 0`; support mismatch → `inf` |
| 2 | `kl_from_logits(p_logits, q_logits)` | Working in log space; `log_softmax`; stability |
| 3 | `my_kl_div(input, target, reduction, log_target)` | Re-implementing `F.kl_div`: argument order and `batchmean` |
| 4 | `js_divergence(p_logits, q_logits)` | A symmetric, bounded divergence; log-space mixtures |
| 5 | `fit_gaussian(target_logp, grid, direction)` | Forward vs reverse KL: mode-covering vs mode-seeking |
| 6 | `kl_gaussian(mu_p, logvar_p, mu_q, logvar_q)` | Closed form for diagonal Gaussians; the VAE term |
| 7 | `masked_token_kl(p_logits, q_logits, mask)` | Token-level KL for LLMs with padding (distillation, RLHF) |
| 8 | `kl_estimators(logp, logq)` | Monte Carlo k1 / k2 / k3 from samples (PPO, GRPO) |
| 9 | `kl_estimator_grad(logits, ref_logits, estimator)` | Which estimator, used as a loss, has the right gradient |

---

### Constraints

- ✅ Distributions live on the **last** dimension; keep every leading (batch) dimension unless told otherwise.
- ✅ Work in log space wherever you can.
- ❌ Do **not** call `F.kl_div`, `nn.KLDivLoss`, `torch.distributions.kl_divergence` or any other KL helper inside your implementations. The tests use them as oracles.
- ❌ No Python loops over batch elements, tokens or classes.

---

<details>
  <summary>💡 Hint</summary>

  - Everything reduces to `Σ p · (log p − log q)`. The parts differ in where `p`, `log p` and `log q` come from.
  - `torch.xlogy(x, y)` computes `x · log(y)` and returns 0 wherever `x == 0`.
  - `F.log_softmax` and `torch.logsumexp` do the max-subtraction trick for you.
  - The k1/k2/k3 estimators come from John Schulman's note *"Approximating KL Divergence"* (2020).

</details>

---

In [ ]:
import math
import torch
import torch.nn.functional as F
from torch.distributions import Categorical, Normal, kl_divergence  # oracles for the tests only

torch.manual_seed(0)

## Part 1: KL between two categorical distributions (probabilities in)

Implement `kl_categorical(p, q)`. Both inputs are probability tensors of shape `(..., K)` that sum to 1 over the last dim. Return KL(P‖Q) with shape `(...)`.

The edge cases are the point:

| case | contribution | why |
|---|---|---|
| `p_k = 0` | `0` | `lim x→0 x·log x = 0`. Naive `p * torch.log(p / q)` gives `0 · (-inf) = nan` |
| `p_k > 0`, `q_k = 0` | `+inf` | Q says impossible something that P says happens |
| `p_k = 0`, `q_k = 0` | `0` | Neither distribution can produce it |

Your function should return `inf`, never `nan`.

In [ ]:
def kl_categorical(p: torch.Tensor, q: torch.Tensor) -> torch.Tensor:
    """
    KL(P || Q) for categorical distributions given as probabilities.

    Args:
        p, q: probabilities of shape (..., K), each summing to 1 over the last dim.

    Returns:
        Tensor of shape (...): sum_k p_k * (log p_k - log q_k), with 0 * log 0 = 0.
    """
    # xlogy(0, anything) == 0, so p == 0 entries vanish instead of producing 0 * -inf = nan.
    # p > 0 with q == 0 gives xlogy(p, 0) = -inf, i.e. KL = +inf, which is the right answer.
    return (torch.xlogy(p, p) - torch.xlogy(p, q)).sum(-1)

In [ ]:
# === Part 1 tests ===
p = torch.softmax(torch.randn(4, 6), -1)
q = torch.softmax(torch.randn(4, 6), -1)
got = kl_categorical(p, q)
ref = kl_divergence(Categorical(probs=p), Categorical(probs=q))
assert got.shape == (4,), f"expected shape (4,), got {tuple(got.shape)}"
assert torch.allclose(got, ref, atol=1e-6), f"mismatch vs torch.distributions: {got} vs {ref}"
assert torch.allclose(kl_categorical(p, p), torch.zeros(4), atol=1e-6), "KL(P||P) must be 0"
assert (got >= 0).all(), "KL is never negative (Gibbs' inequality)"
assert not torch.allclose(got, kl_categorical(q, p)), "KL is not symmetric; check the argument order"

# Zeros in the support
p0 = torch.tensor([0.5, 0.5, 0.0])
q0 = torch.tensor([0.25, 0.25, 0.5])
assert torch.isclose(kl_categorical(p0, q0), torch.tensor(math.log(2.0))), \
    f"p = 0 entries must contribute 0; expected log 2, got {kl_categorical(p0, q0)}"
assert torch.isinf(kl_categorical(q0, p0)), "q = 0 where p > 0 must give +inf (not nan)"
print("Part 1 passed ✓")

## Part 2: KL from logits (the version you'll actually use)

Models output **logits**. Implement `kl_from_logits(p_logits, q_logits)` = KL(softmax(p_logits) ‖ softmax(q_logits)), with output shape `(...)`.

Why not just `kl_categorical(p_logits.softmax(-1), q_logits.softmax(-1))`? With confident (large) logits, softmax **underflows to exactly 0**, `log 0 = -inf`, and you get `inf` or `nan` even though the true KL is finite.

Instead, get `log P` and `log Q` directly with `F.log_softmax`, which uses `logsumexp` (the max-subtraction trick) and never takes the log of an underflowed probability:

$$\mathrm{KL} = \sum_k e^{\log P_k}\,\big(\log P_k - \log Q_k\big)$$

In [ ]:
def kl_from_logits(p_logits: torch.Tensor, q_logits: torch.Tensor) -> torch.Tensor:
    """
    KL(P || Q) where P = softmax(p_logits), Q = softmax(q_logits) over the last dim.

    Args:
        p_logits, q_logits: unnormalized logits of shape (..., K).

    Returns:
        Tensor of shape (...).
    """
    logp = F.log_softmax(p_logits, dim=-1)
    logq = F.log_softmax(q_logits, dim=-1)
    # log-probs are finite for finite logits, so no log(0) ever happens here
    return (logp.exp() * (logp - logq)).sum(-1)

In [ ]:
# === Part 2 tests ===
pl, ql = torch.randn(3, 5, 10), torch.randn(3, 5, 10)
got = kl_from_logits(pl, ql)
ref = kl_divergence(Categorical(logits=pl), Categorical(logits=ql))
assert got.shape == (3, 5), f"expected shape (3, 5), got {tuple(got.shape)}"
assert torch.allclose(got, ref, atol=1e-5), "mismatch vs torch.distributions"
assert torch.allclose(kl_from_logits(pl + 7.0, ql - 3.0), got, atol=1e-5), \
    "softmax is shift-invariant, so adding a constant to the logits must not change the KL"

# Confident logits: the naive probability route breaks, the log-space route doesn't
big_p, big_q = 100 * pl, 100 * ql
naive = kl_categorical(big_p.softmax(-1), big_q.softmax(-1))
stable = kl_from_logits(big_p, big_q)
print(f"naive route: {int((~torch.isfinite(naive)).sum())} of {naive.numel()} entries are inf/nan")
assert torch.isfinite(stable).all(), "kl_from_logits must stay finite for large logits; use log_softmax"
assert (stable >= -1e-4).all(), "KL must be non-negative"
print("Part 2 passed ✓")

## Part 3: Re-implement `F.kl_div` (the API everyone gets wrong)

```python
F.kl_div(input, target, reduction="mean", log_target=False)
```

- `input` is the **model's log-probabilities**, `log Q`. It is *not* probabilities.
- `target` is the **reference distribution P**: probabilities, or `log P` if `log_target=True`.
- The pointwise loss is `P · (log P − input)`, with 0 where `P = 0`.
- So `F.kl_div(log_q, p)` computes **KL(P ‖ Q)**: the second argument is the first argument of the KL.

The reductions:

| `reduction` | returns |
|---|---|
| `'none'` | the pointwise tensor, same shape as `input` |
| `'sum'` | sum of everything |
| `'mean'` | mean over **all elements**, i.e. divided by `B·K`, **not** the KL |
| `'batchmean'` | `sum / input.size(0)`, the true mean KL per example |

Implement `my_kl_div` so that it matches `F.kl_div` for all four reductions and both values of `log_target`. Then read the test output to see how far off `'mean'` is.

> PyTorch warns that `'mean'` will be changed to behave like `'batchmean'` "in the next major release". Through torch 2.x it still divides by `B·K`, so always pass `reduction='batchmean'` explicitly.

In [ ]:
def my_kl_div(input: torch.Tensor, target: torch.Tensor,
              reduction: str = "mean", log_target: bool = False) -> torch.Tensor:
    """
    Same semantics as torch.nn.functional.kl_div.

    Args:
        input:  log-probabilities of the model distribution Q, any shape.
        target: P as probabilities (log_target=False) or log-probabilities (log_target=True).
        reduction: 'none' | 'sum' | 'mean' | 'batchmean'.
        log_target: whether target is given in log space.
    """
    if log_target:
        pointwise = target.exp() * (target - input)
    else:
        pointwise = torch.xlogy(target, target) - target * input

    if reduction == "none":
        return pointwise
    if reduction == "sum":
        return pointwise.sum()
    if reduction == "mean":          # mean over every element; NOT the KL
        return pointwise.mean()
    if reduction == "batchmean":     # what the math actually means
        return pointwise.sum() / input.size(0)
    raise ValueError(f"unknown reduction '{reduction}'")

In [ ]:
# === Part 3 tests ===
B, K = 8, 5
log_q = F.log_softmax(torch.randn(B, K), -1)
p = torch.softmax(torch.randn(B, K), -1)
p_pos = p.clone()                             # strictly positive: log P is finite
p[0, 0] = 0.0
p[0] = p[0] / p[0].sum()                      # a probability target with an exact zero
for log_target in (False, True):
    tgt = p_pos.log() if log_target else p
    for red in ("none", "sum", "mean", "batchmean"):
        got = my_kl_div(log_q, tgt, reduction=red, log_target=log_target)
        exp = F.kl_div(log_q, tgt, reduction=red, log_target=log_target)
        assert torch.allclose(got, exp, atol=1e-6), \
            f"reduction={red}, log_target={log_target}: {got} vs F.kl_div {exp}"

true_kl = kl_categorical(p, log_q.exp()).mean()
print(f"true mean KL(P||Q)      = {true_kl:.5f}")
print(f"F.kl_div 'batchmean'    = {F.kl_div(log_q, p, reduction='batchmean'):.5f}")
print(f"F.kl_div 'mean' default = {F.kl_div(log_q, p):.5f}   <- off by a factor of K = {K}")
assert torch.allclose(my_kl_div(log_q, p, reduction="batchmean"), true_kl, atol=1e-6)
print("Part 3 passed ✓")

## Part 4: Jensen–Shannon divergence (symmetric and bounded)

$$\mathrm{JS}(P, Q) = \tfrac12\,\mathrm{KL}(P\,\|\,M) + \tfrac12\,\mathrm{KL}(Q\,\|\,M), \qquad M = \tfrac12(P + Q)$$

Unlike KL, JS is **symmetric**, **bounded** in `[0, log 2]`, and **finite even when the supports don't overlap**, because M covers both. It is the divergence behind the original GAN objective, and its generalized (β-weighted) form is the loss in GKD distillation.

Implement `js_divergence(p_logits, q_logits)`, with output shape `(...)`. Compute `log M` in log space: `log M = logsumexp([log P, log Q]) − log 2`.

In [ ]:
def js_divergence(p_logits: torch.Tensor, q_logits: torch.Tensor) -> torch.Tensor:
    """
    Jensen-Shannon divergence between softmax(p_logits) and softmax(q_logits).

    Args:
        p_logits, q_logits: logits of shape (..., K).

    Returns:
        Tensor of shape (...), in [0, log 2].
    """
    logp = F.log_softmax(p_logits, dim=-1)
    logq = F.log_softmax(q_logits, dim=-1)
    # log((P + Q) / 2) without leaving log space
    logm = torch.logsumexp(torch.stack([logp, logq]), dim=0) - math.log(2.0)
    kl_pm = (logp.exp() * (logp - logm)).sum(-1)
    kl_qm = (logq.exp() * (logq - logm)).sum(-1)
    return 0.5 * kl_pm + 0.5 * kl_qm

In [ ]:
# === Part 4 tests ===
pl, ql = torch.randn(6, 7), torch.randn(6, 7)
got = js_divergence(pl, ql)
P, Q = pl.softmax(-1), ql.softmax(-1)
M = Categorical(probs=0.5 * (P + Q))
ref = 0.5 * kl_divergence(Categorical(probs=P), M) + 0.5 * kl_divergence(Categorical(probs=Q), M)
assert got.shape == (6,), f"expected shape (6,), got {tuple(got.shape)}"
assert torch.allclose(got, ref, atol=1e-5), "mismatch vs 0.5*KL(P||M) + 0.5*KL(Q||M)"
assert torch.allclose(got, js_divergence(ql, pl), atol=1e-6), "JS must be symmetric"
assert (got >= -1e-6).all() and (got <= math.log(2) + 1e-6).all(), "JS must lie in [0, log 2]"

# Disjoint supports: KL explodes, JS saturates at log 2
far_p = torch.tensor([50.0, -50.0, -50.0, -50.0])
far_q = torch.tensor([-50.0, -50.0, -50.0, 50.0])
print(f"KL = {kl_from_logits(far_p, far_q):.1f}   JS = {js_divergence(far_p, far_q):.4f}   log 2 = {math.log(2):.4f}")
assert abs(js_divergence(far_p, far_q).item() - math.log(2)) < 1e-4, "disjoint supports must give JS = log 2"
print("Part 4 passed ✓")

## Part 5: Forward vs reverse KL (mode-covering vs mode-seeking)

Fit a **single** Gaussian `q_θ` (parameters `μ`, `σ`) to a **bimodal** target `p` in two ways:

| objective | penalizes | behaviour | where you meet it |
|---|---|---|---|
| **forward** KL(p ‖ q_θ) = E_p[log p − log q] | `q` small where `p` is large | **mode-covering**: stretches over both modes | MLE, cross-entropy, SFT, classic distillation |
| **reverse** KL(q_θ ‖ p) = E_q[log q − log p] | `q` large where `p` is small | **mode-seeking**: hides inside one mode | variational inference, KL-regularized RL, MiniLLM |

To keep everything categorical, discretize the real line on a `grid`. `q_θ`'s logits on the grid are `-½((grid − μ)/σ)²`; `log_softmax` normalizes them for you. Then reuse `kl_from_logits`, remembering that it takes **logits**, and that `target_logp` is already a valid set of logits.

Implement `fit_gaussian`: optimize `μ` and `log σ` (so σ stays positive) with Adam, starting from `mu_init` and `sigma_init`, and return `(μ, σ)` as Python floats.

Predict the two answers before you run the test.

In [ ]:
def fit_gaussian(target_logp: torch.Tensor, grid: torch.Tensor, direction: str,
                 steps: int = 500, lr: float = 0.05,
                 mu_init: float = 1.0, sigma_init: float = 0.5) -> tuple:
    """
    Fit a discretized Gaussian q to a target distribution by minimizing KL.

    Args:
        target_logp: log-probabilities of the target on the grid, shape (G,).
        grid: grid points, shape (G,).
        direction: 'forward' -> minimize KL(target || q); 'reverse' -> minimize KL(q || target).
        steps, lr: Adam iterations and learning rate.
        mu_init, sigma_init: starting point.

    Returns:
        (mu, sigma) as Python floats.
    """
    if direction not in ("forward", "reverse"):
        raise ValueError(f"direction must be 'forward' or 'reverse', got '{direction}'")
    mu = torch.tensor(float(mu_init), requires_grad=True)
    log_sigma = torch.tensor(math.log(sigma_init), requires_grad=True)
    opt = torch.optim.Adam([mu, log_sigma], lr=lr)
    for _ in range(steps):
        q_logits = -0.5 * ((grid - mu) / log_sigma.exp()) ** 2
        if direction == "forward":
            loss = kl_from_logits(target_logp, q_logits)
        else:
            loss = kl_from_logits(q_logits, target_logp)
        opt.zero_grad()
        loss.backward()
        opt.step()
    return mu.item(), log_sigma.exp().item()

In [ ]:
# === Part 5 tests ===
grid = torch.linspace(-6, 6, 241)
# Two well-separated modes at -2.5 and +2.5, each with std 0.6. Built in log space so the
# valley between the modes stays finite (in probability space it would underflow to 0).
mixture = torch.logsumexp(torch.stack([Normal(-2.5, 0.6).log_prob(grid),
                                       Normal(2.5, 0.6).log_prob(grid)]), dim=0)
target_logp = F.log_softmax(mixture, dim=-1)

mu_f, sigma_f = fit_gaussian(target_logp, grid, "forward")
mu_r, sigma_r = fit_gaussian(target_logp, grid, "reverse")
print(f"forward KL(p||q): mu = {mu_f:+.2f}, sigma = {sigma_f:.2f}   (covers both modes)")
print(f"reverse KL(q||p): mu = {mu_r:+.2f}, sigma = {sigma_r:.2f}   (locks onto one mode)")

assert abs(mu_f) < 0.5 and sigma_f > 2.0, "forward KL should centre between the modes with a wide sigma"
assert abs(mu_r - 2.5) < 0.3 and sigma_r < 1.0, "reverse KL should sit on the +2.5 mode with a narrow sigma"
print("Part 5 passed ✓")

> **Try it:** rerun the reverse fit with `mu_init=0.5, sigma_init=1.0`. It ends up wide and centred, stuck between the modes. Reverse KL is non-convex in θ, so *where you start decides which mode, or which compromise, you get*. That's one reason RL fine-tuning with a KL penalty is sensitive to initialization (the SFT model).

## Part 6: Closed-form KL between diagonal Gaussians (the VAE term)

For continuous distributions the sum becomes an integral. For two 1-D Gaussians it has a closed form:

$$\mathrm{KL}\big(\mathcal N(\mu_p,\sigma_p^2)\,\|\,\mathcal N(\mu_q,\sigma_q^2)\big) = \log\frac{\sigma_q}{\sigma_p} + \frac{\sigma_p^2 + (\mu_p-\mu_q)^2}{2\sigma_q^2} - \frac12$$

For **diagonal** Gaussians in `D` dimensions, the KL is the sum over dimensions.

Implement `kl_gaussian` taking **log-variances** (that's what an encoder outputs, because it keeps σ² > 0 for free). Reduce over the last dim.

Then check the special case every VAE uses, with `q = N(0, I)`:

$$\mathrm{KL}\big(\mathcal N(\mu,\sigma^2)\,\|\,\mathcal N(0, I)\big) = -\tfrac12\sum_d\big(1 + \log\sigma_d^2 - \mu_d^2 - \sigma_d^2\big)$$

In [ ]:
def kl_gaussian(mu_p: torch.Tensor, logvar_p: torch.Tensor,
                mu_q: torch.Tensor, logvar_q: torch.Tensor) -> torch.Tensor:
    """
    KL( N(mu_p, diag(exp(logvar_p))) || N(mu_q, diag(exp(logvar_q))) ).

    Args:
        mu_p, logvar_p, mu_q, logvar_q: tensors of shape (..., D) (broadcastable).

    Returns:
        Tensor of shape (...): KL summed over the D dimensions.
    """
    # log(sigma_q / sigma_p) = 0.5 * (logvar_q - logvar_p); dividing by var_q is a
    # multiplication by exp(-logvar_q), so no explicit sigma is ever formed.
    per_dim = 0.5 * (logvar_q - logvar_p
                     + (logvar_p.exp() + (mu_p - mu_q) ** 2) * torch.exp(-logvar_q)
                     - 1.0)
    return per_dim.sum(-1)

In [ ]:
# === Part 6 tests ===
mu_p, lv_p = torch.randn(4, 3), torch.randn(4, 3)
mu_q, lv_q = torch.randn(4, 3), torch.randn(4, 3)
got = kl_gaussian(mu_p, lv_p, mu_q, lv_q)
ref = kl_divergence(Normal(mu_p, (0.5 * lv_p).exp()), Normal(mu_q, (0.5 * lv_q).exp())).sum(-1)
assert got.shape == (4,), f"expected shape (4,), got {tuple(got.shape)}"
assert torch.allclose(got, ref, atol=1e-5), "mismatch vs torch.distributions.Normal"
assert torch.allclose(kl_gaussian(mu_p, lv_p, mu_p, lv_p), torch.zeros(4), atol=1e-6), "KL(p||p) must be 0"

# The VAE special case: q = N(0, I)
zeros = torch.zeros_like(mu_p)
vae = -0.5 * (1 + lv_p - mu_p ** 2 - lv_p.exp()).sum(-1)
assert torch.allclose(kl_gaussian(mu_p, lv_p, zeros, zeros), vae, atol=1e-5), "VAE closed form mismatch"
print("Part 6 passed ✓")

## Part 7: Token-level KL for language models (with padding)

In LLM distillation and RLHF you compare two models **at every position** of a batch of sequences:

- logits `(B, T, V)` from each model,
- a `mask` of shape `(B, T)`: 1 for real tokens, 0 for padding (or for prompt tokens you don't want to score).

Implement `masked_token_kl(p_logits, q_logits, mask)`. It should:

1. compute the exact, full-vocabulary KL(P_t ‖ Q_t) at every position,
2. **ignore** masked positions completely, even if their logits are garbage, and
3. return a scalar: the **mean over unmasked tokens** (not over sequences).

Directions in practice:

- **forward distillation**: `masked_token_kl(teacher_logits.detach(), student_logits, mask)`
- **reverse distillation / RL penalty**: `masked_token_kl(student_logits, teacher_logits.detach(), mask)`

In [ ]:
def masked_token_kl(p_logits: torch.Tensor, q_logits: torch.Tensor,
                    mask: torch.Tensor) -> torch.Tensor:
    """
    Mean per-token KL(P_t || Q_t) over unmasked positions.

    Args:
        p_logits, q_logits: logits of shape (B, T, V).
        mask: (B, T), 1 (or True) for tokens to include, 0 for padding.

    Returns:
        Scalar tensor.
    """
    per_token = kl_from_logits(p_logits, q_logits)          # (B, T)
    mask = mask.to(per_token.dtype)
    # Multiplying by 0 is safe because per_token is finite for finite logits; with -inf
    # logits you'd need torch.where to keep a nan at a pad position out of the sum.
    return (per_token * mask).sum() / mask.sum().clamp(min=1.0)

In [ ]:
# === Part 7 tests ===
B, T, V = 3, 6, 11
pl, ql = torch.randn(B, T, V), torch.randn(B, T, V)
pl[2] *= 4                                    # the short sequence disagrees much more
lengths = [6, 4, 2]
mask = torch.zeros(B, T)
for i, n in enumerate(lengths):
    mask[i, :n] = 1

got = masked_token_kl(pl, ql, mask)
oracle = torch.stack([kl_divergence(Categorical(logits=pl[i, t]), Categorical(logits=ql[i, t]))
                      for i in range(B) for t in range(lengths[i])]).mean()
assert got.ndim == 0, f"expected a scalar, got shape {tuple(got.shape)}"
assert torch.allclose(got, oracle, atol=1e-5), f"expected the mean over the {sum(lengths)} real tokens"

# Garbage at padded positions must not leak in
pl_garbage, ql_garbage = pl.clone(), ql.clone()
pl_garbage[mask == 0] = 1e3 * torch.randn(int((mask == 0).sum()), V)
ql_garbage[mask == 0] = -1e3 * torch.randn(int((mask == 0).sum()), V)
assert torch.allclose(masked_token_kl(pl_garbage, ql_garbage, mask), got, atol=1e-5), \
    "changing logits at padded positions changed the result"

# Gradients reach real tokens only
qg = ql.clone().requires_grad_(True)
masked_token_kl(pl, qg, mask).backward()
assert qg.grad[mask == 0].abs().max() == 0, "padded positions must get zero gradient"
assert qg.grad[mask == 1].abs().max() > 0, "real positions must get a gradient"

# Token mean, not sequence mean
seq_mean = torch.stack([kl_from_logits(pl[i, :n], ql[i, :n]).mean() for i, n in enumerate(lengths)]).mean()
print(f"token mean = {got:.4f}   sequence mean = {seq_mean:.4f}")
print("(the 2-token sequence is 2/12 of the token mean but 1/3 of the sequence mean)")
print("Part 7 passed ✓")

## Part 8: Estimating KL from samples (k1, k2, k3)

In RL fine-tuning of LLMs, the exact KL over whole sequences is a sum over `V^T` possibilities, which is impossible. Often all you have is the **log-prob of the token that was actually sampled**, under the policy and under the reference model. So you estimate KL(π ‖ π_ref) by Monte Carlo, with `x ~ π`.

Let `logp = log π(x)` and `logq = log π_ref(x)`, and define the log-ratio `log r = logq − logp`. Schulman's three estimators are:

| estimator | formula | unbiased? | sign | variance |
|---|---|---|---|---|
| **k1** | `−log r` | ✅ | can be **negative** | high |
| **k2** | `½ (log r)²` | ❌ (slightly biased) | ≥ 0 | low |
| **k3** | `(r − 1) − log r` | ✅ | ≥ 0 | low |

- **Why k3 is unbiased:** `E_{x~π}[r] = Σ π · (π_ref/π) = 1`, so `(r − 1)` is a zero-mean **control variate** added to k1.
- **Why k3 is ≥ 0:** `log r ≤ r − 1` for every r.

PPO-RLHF uses k1 per token. GRPO uses k3.

Implement `kl_estimators(logp, logq)`, returning the per-sample tuple `(k1, k2, k3)`, each with the same shape as the inputs. Then check the bias and variance experimentally.

In [ ]:
def kl_estimators(logp: torch.Tensor, logq: torch.Tensor) -> tuple:
    """
    Per-sample Monte Carlo estimators of KL(p || q), for samples x ~ p.

    Args:
        logp: log p(x) of the sampled x (the sampling distribution, e.g. the policy).
        logq: log q(x) of the same x (e.g. the reference model). Same shape as logp.

    Returns:
        (k1, k2, k3), each the same shape as logp.
    """
    log_ratio = logq - logp                       # log r, r = q(x) / p(x)
    k1 = -log_ratio
    k2 = 0.5 * log_ratio ** 2
    # torch.expm1(x) = exp(x) - 1, accurate when r is close to 1 (the usual case)
    k3 = torch.expm1(log_ratio) - log_ratio
    return k1, k2, k3

In [ ]:
# === Part 8 tests ===
torch.manual_seed(0)
N = 200_000
print(f"{'setting':>8} {'true KL':>9} | {'k1 bias/KL':>10} {'k1 std/KL':>9} | "
      f"{'k2 bias/KL':>10} {'k2 std/KL':>9} | {'k3 bias/KL':>10} {'k3 std/KL':>9}")
for name, scale in (("close", 0.1), ("far", 1.0)):
    policy_logits = torch.randn(50)
    ref_logits = policy_logits + scale * torch.randn(50)
    true = kl_from_logits(policy_logits, ref_logits).item()
    x = Categorical(logits=policy_logits).sample((N,))            # samples from the policy
    logp = F.log_softmax(policy_logits, -1)[x]
    logq = F.log_softmax(ref_logits, -1)[x]
    k1, k2, k3 = kl_estimators(logp, logq)
    row = [f"{(k.mean().item() - true) / true:>10.3f} {k.std().item() / true:>9.2f}" for k in (k1, k2, k3)]
    print(f"{name:>8} {true:>9.5f} | " + " | ".join(row))

    se = lambda k: k.std().item() / math.sqrt(N)
    assert abs(k1.mean().item() - true) < 5 * se(k1), "k1 should be unbiased"
    assert abs(k3.mean().item() - true) < 5 * se(k3), "k3 should be unbiased"
    assert (k2 >= 0).all() and (k3 >= -1e-6).all(), "k2 and k3 are non-negative per sample"
    assert (k1 < 0).any(), "k1 should produce some negative per-sample estimates"
    assert k3.std() < k1.std(), "k3 should have lower variance than k1"
    if name == "far":
        assert k2.mean().item() - true > 0.1 * true, "k2's bias should show up when p and q are far apart"
print("Part 8 passed ✓")

## Part 9: KL as a loss. Does the estimator have the right gradient?

Part 8 was about **values**. In training you **differentiate**. A common pattern (GRPO is one example) is to add `β · mean(k(x))` straight to the loss, with the samples `x ~ π_θ` treated as fixed data. Autograd then differentiates `k` *only through `log π_θ(x)`*, never through the sampling. In expectation the gradient you get is

$$g_k = \mathbb{E}_{x\sim\pi_\theta}\big[\nabla_\theta\, k(x;\theta)\big].$$

Is `g_k` equal to `∇_θ KL(π_θ ‖ π_ref)`?

For a single categorical "policy" with logits θ, you can compute that expectation **exactly**, without sampling noise, by weighting every category by its **detached** probability:

$$\text{surrogate} = \sum_x \mathrm{sg}\big(\pi_\theta(x)\big)\, k(x;\theta), \qquad g_k = \nabla_\theta\,\text{surrogate}$$

where `sg` is stop-gradient (`.detach()`).

Implement `kl_estimator_grad(logits, ref_logits, estimator)`, which returns `g_k` w.r.t. `logits` (same shape as `logits`) for `estimator` ∈ `{'k1', 'k2', 'k3'}`. The test compares each one against autograd of the exact **reverse** KL(π‖π_ref) and the exact **forward** KL(π_ref‖π).

**Write down your prediction for each of the three before running the test.**

In [ ]:
def kl_estimator_grad(logits: torch.Tensor, ref_logits: torch.Tensor, estimator: str) -> torch.Tensor:
    """
    Expected gradient (w.r.t. logits) of an on-policy KL estimator used as a loss.

    Args:
        logits: policy logits, shape (..., V).
        ref_logits: reference logits, shape (..., V); no gradient flows into them.
        estimator: 'k1', 'k2' or 'k3' (see kl_estimators).

    Returns:
        Tensor with the same shape as logits: grad of sum_x sg(pi(x)) * k(x) w.r.t. logits.
    """
    index = {"k1": 0, "k2": 1, "k3": 2}[estimator]
    logits = logits.detach().requires_grad_(True)
    logp = F.log_softmax(logits, dim=-1)
    logq = F.log_softmax(ref_logits, dim=-1).detach()
    k = kl_estimators(logp, logq)[index]
    # Exact expectation over x ~ pi with pi detached: what autograd sees when the samples
    # are fixed data and only log pi(x) carries gradient.
    surrogate = (logp.exp().detach() * k).sum()
    (grad,) = torch.autograd.grad(surrogate, logits)
    return grad

In [ ]:
# === Part 9 tests ===
torch.manual_seed(0)
policy_logits, ref_logits = torch.randn(4, 10), torch.randn(4, 10)

theta = policy_logits.clone().requires_grad_(True)
grad_reverse = torch.autograd.grad(kl_from_logits(theta, ref_logits).sum(), theta)[0]   # KL(pi || ref)
grad_forward = torch.autograd.grad(kl_from_logits(ref_logits, theta).sum(), theta)[0]   # KL(ref || pi)

grads = {e: kl_estimator_grad(policy_logits, ref_logits, e) for e in ("k1", "k2", "k3")}
print(f"{'as loss':>8} | {'|g|':>7} | {'max |g - grad reverse KL|':>26} | {'max |g - grad forward KL|':>26}")
for e, g in grads.items():
    assert g.shape == policy_logits.shape, f"{e}: expected shape {tuple(policy_logits.shape)}"
    print(f"{e:>8} | {g.norm():7.4f} | {(g - grad_reverse).abs().max():26.2e} | {(g - grad_forward).abs().max():26.2e}")

assert torch.allclose(grads["k1"], torch.zeros_like(policy_logits), atol=1e-6), \
    "k1 as a loss: E[grad log pi] = 0, so its expected gradient should vanish"
assert torch.allclose(grads["k2"], grad_reverse, atol=1e-5), \
    "k2 as a loss should reproduce the gradient of the reverse KL(pi || ref)"
assert torch.allclose(grads["k3"], grad_forward, atol=1e-5), \
    "k3 as a loss: its expected gradient is that of the forward KL(ref || pi)"
print("Part 9 passed ✓")

**What just happened.** With samples treated as data, the gradient of each estimator is `c(x)·∇log π(x)`:

| used as a loss | per-sample gradient | expected gradient |
|---|---|---|
| k1 | `∇log π` | **0**, because `E_π[∇log π] = Σ ∇π = ∇1 = 0` |
| k2 | `(log π − log π_ref)·∇log π` | ∇ **reverse** KL(π‖π_ref), exactly |
| k3 | `(1 − π_ref/π)·∇log π` | `−E_{π_ref}[∇log π]` = ∇ **forward** KL(π_ref‖π) |

So **an unbiased estimator of the value (k1, k3) is not automatically a correct loss for the gradient**. The PPO-RLHF recipe, which puts k1 **in the reward** (detached, multiplied by the score function), has the same gradient as k2-as-loss. Both are correct for reverse KL. GRPO's k3-as-loss matches it only to first order near π = π_ref.

## Interview follow-ups (with answers)

1. **`F.kl_div(input, target)` computes KL(target ‖ exp(input)).** `input` must be the *model's* log-probabilities (log Q), and `target` is the *reference* P, as probabilities by default or as log-probs with `log_target=True`. The order is the opposite of the math notation, which is the classic bug. `reduction='mean'` averages over **every element**, so it returns `batchmean / K`, which is not a KL value at all. Use `reduction='batchmean'`. PyTorch's docs say this explicitly, and `'mean'` stays the default only for consistency with the other losses.

2. **Forward KL from the data to the model.** Cross-entropy is `H(P_data, Q_θ) = H(P_data) + KL(P_data ‖ Q_θ)`, and `H(P_data)` doesn't depend on θ. So maximum likelihood, SFT and classification all minimize KL(data ‖ model). That makes them **mode-covering**: the model is punished hard for assigning low probability to anything the data contains.

3. **Usually reverse KL (or something between the two).** Forward KL(teacher ‖ student) forces a small student to spread mass over everything the teacher might say, including the teacher's low-probability tail. At generation time that produces samples the teacher would never produce. Reverse KL(student ‖ teacher) is mode-seeking: the student commits to the teacher's major modes. **MiniLLM** (Gu et al., 2023) minimizes reverse KL with policy-gradient-style optimization on student samples. **GKD** (Agarwal et al., 2024) trains on the student's own samples, either fully or as a mixed fraction, with a generalized Jensen–Shannon divergence whose β interpolates between the forward and reverse directions. Part 5 is the 1-D version of this picture.

4. **Gradient scale.** With soft targets at temperature `T`, the gradient of the soft loss w.r.t. the logits scales like `1/T²`. Multiplying by `T²` keeps its magnitude comparable to the hard-label term as `T` changes. See the `knowledge-distillation` problem.

5. **Not exactly.** Part 9 shows that, on-policy:
   - **k1 in the reward** (detached, multiplied by the score function) gives the exact gradient of **reverse** KL(π‖π_ref). PPO-RLHF does this, per token.
   - **k2 as a loss** has the *same* gradient, `(log π − log π_ref)·∇log π`.
   - **k1 as a loss** has expected gradient **zero**, because `E_π[∇log π] = 0`. It does nothing.
   - **k3 as a loss** (GRPO) has expected gradient `E_π[(1 − π_ref/π)∇log π] = −E_{π_ref}[∇log π]`, which is the gradient of the **forward** KL(π_ref‖π). Forward and reverse KL agree to second order near π = π_ref, so k3-as-loss is a first-order approximation of the reverse-KL gradient that drifts as the policy moves away.

   Liu et al. (2025), *"Rethinking KL Regularization in RLHF"*, make the same equivalences precise.

6. With `logits = -inf`, `log_softmax` gives `-inf`. If **both** distributions are `-inf` at the same index, `logp − logq = -inf − (-inf) = nan`, and `0 · nan = nan`. If only Q is `-inf` where P > 0, the KL really is `inf`. Fix: compute the pointwise term with `torch.where(p > 0, p * (logp - logq), 0)`, or mask the filtered vocabulary out before reducing, or use a large finite negative value (e.g. `torch.finfo(dtype).min`) instead of `-inf`.

7. **Full-covariance Gaussians:** `KL = ½[ tr(Σ_q⁻¹Σ_p) + (μ_q−μ_p)ᵀΣ_q⁻¹(μ_q−μ_p) − k + log det Σ_q − log det Σ_p ]`. Never form an inverse: take Cholesky factors `L_q`, use `torch.linalg.solve_triangular` for both the trace term and the Mahalanobis term, and get `log det Σ = 2 Σ log diag(L)`.

8. A **token mean** weights every token equally. A **sequence mean** (average within each sequence, then across sequences) gives each token in a short sequence more weight than a token in a long one, which introduces a length bias. Most LLM losses use the token mean. Recent RL recipes such as DAPO and Dr. GRPO changed their loss aggregation for exactly this reason. Choose a sequence mean only when every *example* should count equally regardless of length.

9. **No.** k3 relies on `E_{x∼π}[π_ref(x)/π(x)] = 1`, which only holds when `x` really is sampled from π. On samples from π_old, you need an importance weight `π/π_old` (usually clipped), or you are estimating a different divergence. The masked mean of k3 is still non-negative, but it no longer estimates either KL direction.